In [1]:
from robot import Robot, MotomanRobot
from planning_scene import PlanningScene
from geometric_trajopt_ipopt import PoseTrajOpt
import numpy as np
import scipy as sp
import transformations as tf


In [2]:
# test the robot with the scene
# add environment collisions
pcd_total = []
# shelf-bottom
num_points = 1000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 1000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 1000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 1000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 1000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


torso_b1 = ["torso_joint_b1"]
left = [
    "arm_left_joint_1_s",
    "arm_left_joint_2_l",
    "arm_left_joint_3_e",
    "arm_left_joint_4_u",
    "arm_left_joint_5_r",
    "arm_left_joint_6_b",
    "arm_left_joint_7_t",
]
right = [
    "arm_right_joint_1_s",
    "arm_right_joint_2_l",
    "arm_right_joint_3_e",
    "arm_right_joint_4_u",
    "arm_right_joint_5_r",
    "arm_right_joint_6_b",
    "arm_right_joint_7_t",
]
robot_joint_names = right#torso_b1 + left + right
robot = MotomanRobot(selected_joint_names=robot_joint_names)
# scene = PlanningScene(robot, scene_pcd=pcd_total)
scene = PlanningScene(robot, scene_pcd=None)
scene.update_scene_pcd(pcd_total)


In [3]:
pos = np.array([0.9096643361780983, -0.22, 1.2246445275392399])
quat = np.array([0.99997882565292, 0.0020695812292800746, -0.005795312726922581, 0.0021164663332217232])
pose = tf.quaternion_matrix(quat)
pose[:3,3] = pos
n_waypoints = 10
start_q = np.zeros((len(robot.selected_joint_dofids)))
solver = PoseTrajOpt(robot, scene, start_q=start_q, target_link="arm_right_link_7_t", target_pose=pose, n_waypoints=n_waypoints)


lb shape:  (70,)
lb: 
[-3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986
 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706
 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619
 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159
 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986
 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159
 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159]
ub: 
[3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986
 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619
 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619 3.14159 1.91986
 3.14159 3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159
 1.91986 2.96706 2.35619 3.14159 1.91986 

In [4]:
qs = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1], size=(2, len(robot_joint_names)))
# qs = np.linspace(start=qs[0,:], stop=qs[1,:], num=n_waypoints+1)
# solver.set_start_q(qs[0,:])
# qs = qs[1:,:]
# print(solver.start_q)
# print(qs)

step = (qs[1]-qs[0]) / (n_waypoints+1)
q0 = qs[0,:]
qs = []
solver.set_start_q(q0)
qs.append(q0+step)
for i in range(n_waypoints-1):
    qs.append(qs[-1]+step)
qs = np.array(qs)

In [5]:
def naive_finite_diff(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # x is a numpy array of shape (n,)
    diff = eps
    grad = np.zeros(x.shape)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += diff
        x_minus = x.copy()
        x_minus[i] -= diff
        grad[i] = (func(x_plus) - func(x_minus)) / (2*diff)
    return grad

def naive_finite_diff_jacobian(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # func returns array of shape (M,)  #(M can be a list)
    # x is a numpy array of shape (n,)
    # result of shape (M,n)
    func_shape = list(func(x).shape)
    x_shape = list(x.shape)
    jacobian = np.zeros(list(func(x).shape)+list(x.shape)).reshape((np.prod(func_shape), np.prod(x_shape)))  # first flatten the func shape
    for i in range(np.prod(x_shape)):
        x_plus = x.copy()
        x_plus.flat[i] += eps
        x_minus = x.copy()
        x_minus.flat[i] -= eps
        
        deriv = (func(x_plus) - func(x_minus)) / (2*eps)
        jacobian[:,i] = deriv.flatten()
    jacobian = jacobian.reshape(func_shape+x_shape)
    return jacobian

In [6]:
# check robot.get_point_on_link_spatial_jacobian
robot.set_selected_joint_values(qs[-1])
point_on_link = np.random.random((3))
pose = robot.get_link_pose(solver.target_link)
point_in_world = pose[:3,:3].dot(point_on_link) + pose[:3,3]
jac = robot.get_point_on_link_spatial_jacobian(solver.target_link, point_on_link)
# obtain the numerical jacobian dp/dq = d(g*p)/dq
def get_point_in_world(q, link_name, point_on_link):
    robot.set_selected_joint_values(q)
    pose = robot.get_link_pose(link_name)
    return pose[:3,:3].dot(point_on_link) + pose[:3,3]

numerical_jacobian = naive_finite_diff_jacobian(lambda q: get_point_in_world(q, solver.target_link, point_on_link), qs[-1], eps=1e-8)
print('diff: ', np.linalg.norm(jac-numerical_jacobian))

diff:  7.830822651031156e-08


In [7]:
scene.visualize()

[TriangleMesh with 194 points and 384 triangles.,
 TriangleMesh with 579 points and 1154 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 10905 points and 21834 triangles.,
 TriangleMesh with 17156 points and 34248 triangles.,
 TriangleMesh with 672 points and 1340 triangles.,
 TriangleMesh with 888 points an

In [8]:
# check the collision constraints
# first obtain the points in the link, record it, and then change the robot configuration, and get the distance of the links again.
def compute_collision_constraints(solver, qs):
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.1
    constrs = []
    jacs = []

    contact_local_pts = []  # ([link1, link2, obj_idx1, obj_idx2, contact_pts_local])
    # contact_pts_local: [(pt1_on_link1, pt2_on_link2)]
    for i in range(solver.n_waypoints):
        contact_local_pts_i = []
        robot.set_selected_joint_values(qs[i])
        # handle self collision
        collision_results = scene.compute_collision_total(security_margin=cc_margin, full=True)
        # if not in collision, then return the min distance
        for col_i in range(len(collision_results)):
            link1, link2, obj_idx1, obj_idx2, collision_result = collision_results[col_i]
            if not collision_result.isCollision():
                # constrs.append(cc_margin)
                continue
            # obtain the pose of the two links
            pose1 = robot.get_link_pose(link1)
            pose1_inv = np.linalg.inv(pose1)
            pose2 = np.eye(4)
            pose2_inv = np.eye(4)
            if link2 != "scene":
                pose2 = robot.get_link_pose(link2)
                pose2_inv = np.linalg.inv(pose2)
            contact_pts_local = []
            # otherwise, compute jacobian. use the mean of each contact point
            dist = 0.0
            # jac = np.zeros((robot.nv, 1))
            jac  = np.zeros((solver.n_waypoints, len(robot.selected_joint_dofids)))
            n_contacts = collision_result.numContacts()
            for contact_i in range(n_contacts):
                contact = collision_result.getContact(contact_i)
                dist += (-contact.penetration_depth)
                # * compute jacobian
                # self collision:
                # d g(q) / d q = 1/g(q)*(J_p1(q) - J_p2(q))
                # NOTE: the point p1 and p2 are in the world frame
                # it seems that the contact point is the intermediate point between the two objects
                normal = np.array(contact.normal)
                pt = np.array(contact.pos)
                p1 = pt - normal * 0.5 * (-contact.penetration_depth)
                p2 = pt + normal * 0.5 * (-contact.penetration_depth)
                # get the point on the link
                p1_local = pose1_inv[:3,:3]@p1 + pose1_inv[:3,3]
                p2_local = pose2_inv[:3,:3]@p2 + pose2_inv[:3,3]
                contact_pts_local.append((p1_local, p2_local))
                if link2 != "scene":
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac2 = robot.get_point_on_link_spatial_jacobian(link2, p2_local)
                    jac_i = np.tensordot(normal, jac2 - jac1, axes=1)
                    # jac_i = normal@(jac1 - jac2)
                else:
                    # collision with environment:
                    # d g(q) / d q = 1/g(q)*J_p1(q)
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac_i = np.tensordot(normal, -jac1, axes=1)
                    # jac_i = normal@jac1
                jac[i] += jac_i

            dist = dist / n_contacts
            jac[i] = jac[i] / n_contacts
            constrs.append(dist)
            jacs.append(jac)
            contact_local_pts_i.append((link1, link2, obj_idx1, obj_idx2, contact_pts_local))
        contact_local_pts.append(contact_local_pts_i)
    constrs = np.array(constrs)
    jacs = np.array(jacs)
    return constrs, jacs, contact_local_pts

In [16]:

def compute_collision_constraints(solver, qs):
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.05
    constrs = []
    jacs = []  # jacobian shape: n_waypoints*n_cols x n_waypoints*ndof
    row = []
    col = []
    contact_local_pts = []  # ([link1, link2, obj_idx1, obj_idx2, contact_pts_local])
    for i in range(solver.n_waypoints):
        # start_time = time.time()
        contact_local_pts_i = []
        robot.set_selected_joint_values(qs[i])
        # handle self collision
        distance_results = scene.compute_collision_min_dist_total(dist_upper_bound=cc_margin, security_margin=0.001, full=True)
        # collision_results = scene.compute_collision_total(security_margin=cc_margin, full=True)
        # print('compute collision time: ', time.time()-start_time)

        # start_time = time.time()  # time for computing jacobian
        # if not in collision, then return the min distance
        for col_i in range(len(distance_results)):
            link1, link2, obj_idx1, obj_idx2, distance_result = distance_results[col_i]
            # obtain the pose of the two links
            pose1 = robot.get_link_pose(link1)
            pose1_inv = np.linalg.inv(pose1)
            pose2 = np.eye(4)
            pose2_inv = np.eye(4)
            if link2 != "scene":
                pose2 = robot.get_link_pose(link2)
                pose2_inv = np.linalg.inv(pose2)

            jac = np.zeros((len(robot.selected_joint_dofids)))
            contact_pts_local = []
            if distance_result['distance'] >= cc_margin:
                dist = cc_margin
                continue
                # constrs.append(cc_margin)
                # continue  # need to compute jacobian as well
            else:
                # dist = 0.0 # at least one contact. So we compute the mean of the contact distance
                dist = distance_result['distance']
                jac = np.zeros((len(robot.selected_joint_dofids)))
                p1 = distance_result['p1']
                p2 = distance_result['p2']
                p1_local = pose1_inv[:3,:3]@p1 + pose1_inv[:3,3]
                p2_local = pose2_inv[:3,:3]@p2 + pose2_inv[:3,3]
                contact_pts_local.append((p1_local, p2_local))
                normal = distance_result['normal']
                print(normal)
                print('p1: ', p1)
                print('p2: ', p2)
                print('norm: ', np.linalg.norm(normal))
                print('normal: ', normal/np.linalg.norm(normal))
                print('normal from distance_result: ', distance_result['normal'])
                print('dist: ', dist)
                # assert np.linalg.norm(normal) == dist
                if link2 != "scene":
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac2 = robot.get_point_on_link_spatial_jacobian(link2, p2_local)
                    jac = np.tensordot(normal, jac2 - jac1, axes=1)
                    # jac_i = normal@(jac1 - jac2)
                else:
                    # collision with environment:
                    # d g(q) / d q = 1/g(q)*J_p1(q)
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac = np.tensordot(normal, -jac1, axes=1)
                    # jac_i = normal@jac1

            constrs.append(dist)
            jacs.append(jac)  # shape: ndof
            row_i = np.zeros(jac.shape) + i*len(distance_results)+col_i # this is the index for the collisions
                                                                            # shape: ndof
            column_i = np.arange(len(jac)) + i*solver.ndof # this is the index for the joint angles
            row.append(row_i)
            col.append(column_i)
            contact_local_pts_i.append((link1, link2, obj_idx1, obj_idx2, contact_pts_local))
        contact_local_pts.append(contact_local_pts_i)
        
    constrs = np.array(constrs) # self.n_waypoints*len(collision_results)
    jacs_data = np.array(jacs).flatten()  # shape: n_waypoints*n_cols x ndof
    row = np.array(row).flatten()
    col = np.array(col).flatten()
    # jacs = sp.sparse.coo_array((jacs_data, (row, col)), shape=(self.n_waypoints*len(collision_results),
    #                                                            self.n_waypoints*self.ndof))
    jacs = jacs_data
    solver.prev_qs = qs.flatten()
    solver.prev_collision_constrs = constrs
    solver.prev_collision_jacs = jacs

    constrs = np.array(constrs)
    jacs = np.array(jacs)
    return constrs, jacs, contact_local_pts


In [17]:
def compute_pt_distance(solver, contact_local_pts, qs):
    # given the local pts in the world frame, and compute the distance between each pair
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.1

    res = []
    # contact_pts_local: [(pt1_on_link1, pt2_on_link2)]
    for i in range(solver.n_waypoints):
        robot.set_selected_joint_values(qs[i])
        contact_local_pts_i = contact_local_pts[i]
        for contact_local_pts_i_contact_k in contact_local_pts_i:
            link1, link2, obj_idx1, obj_idx2, contact_pts_local = contact_local_pts_i_contact_k
            pose1 = robot.get_link_pose(link1)
            pose2 = np.eye(4)
            if link2 != "scene":
                pose2 = robot.get_link_pose(link2)
            dist_avg = 0.0
            # compute the distance between each pair points
            for j in range(len(contact_pts_local)):
                p1_local, p2_local = contact_pts_local[j]
                p1 = pose1[:3,:3]@p1_local + pose1[:3,3]
                p2 = pose2[:3,:3]@p2_local + pose2[:3,3]
                dist = np.linalg.norm(p1-p2)
                dist_avg += dist
            dist_avg = dist_avg / len(contact_pts_local)
            res.append(dist_avg)
    return np.array(res)

In [18]:
constrs, jacs, contact_local_pts = compute_collision_constraints(solver, qs)
# finite differencing
numerical_jacs = naive_finite_diff_jacobian(lambda qs: compute_pt_distance(solver, contact_local_pts, qs), qs, eps=1e-8)

[1.09984391e-04 9.99999966e-01 2.36212650e-04]
p1:  [0.09399509 0.12896494 1.24049159]
p2:  [0.09399999 0.1735     1.24050211]
norm:  0.9999999999999999
normal:  [1.09984391e-04 9.99999966e-01 2.36212650e-04]
normal from distance_result:  [1.09984391e-04 9.99999966e-01 2.36212650e-04]
dist:  0.04453506883924012
[ 1.43239793e-04 -9.99999968e-01  2.10393540e-04]
p1:  [ 0.14272881 -0.12895531  1.25539371]
p2:  [ 0.1427352  -0.17353579  1.25540309]
norm:  1.0
normal:  [ 1.43239793e-04 -9.99999968e-01  2.10393540e-04]
normal from distance_result:  [ 1.43239793e-04 -9.99999968e-01  2.10393540e-04]
dist:  0.04458047790702259
[-0.1538364   0.87907232  0.45118313]
p1:  [-0.13248869 -0.02827987  1.58430961]
p2:  [-0.13375014 -0.02107152  1.58800929]
norm:  1.0
normal:  [-0.1538364   0.87907232  0.45118313]
normal from distance_result:  [-0.1538364   0.87907232  0.45118313]
dist:  0.008199952085343415
[-0.15383634  0.87907235  0.4511831 ]
p1:  [-0.13248869 -0.02827987  1.58430961]
p2:  [-0.134967

In [19]:
print(constrs.shape)
print(jacs.shape)
dist = compute_pt_distance(solver, contact_local_pts, qs)
print(dist.shape)
print('diff: ', np.linalg.norm(constrs-dist))

(132,)
(924,)
(132,)
diff:  1.1247728837987599e-15


In [12]:
print('diff: ', np.linalg.norm(jacs-numerical_jacs))

diff:  5.2078769290949e-07


In [13]:
# check the collision constraints
# first obtain the points in the link, record it, and then change the robot configuration, and get the distance of the links again.
def compute_collision_constraints_single_step(solver, q):
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.1
    constrs = []
    jacs = []

    contact_local_pts = []  # ([link1, link2, obj_idx1, obj_idx2, contact_pts_local])
    # contact_pts_local: [(pt1_on_link1, pt2_on_link2)]
    robot.set_selected_joint_values(q)
    # handle self collision
    collision_results = scene.compute_collision_total(security_margin=cc_margin, full=True)
    # if not in collision, then return the min distance
    for col_i in range(len(collision_results)):
        link1, link2, obj_idx1, obj_idx2, collision_result = collision_results[col_i]
        if not collision_result.isCollision():
            # constrs.append(cc_margin)
            continue
        # obtain the pose of the two links
        pose1 = robot.get_link_pose(link1)
        pose1_inv = np.linalg.inv(pose1)
        pose2 = np.eye(4)
        pose2_inv = np.eye(4)
        if link2 != "scene":
            pose2 = robot.get_link_pose(link2)
            pose2_inv = np.linalg.inv(pose2)
        contact_pts_local = []
        # otherwise, compute jacobian. use the mean of each contact point
        dist = 0.0
        # jac = np.zeros((robot.nv, 1))
        jac  = np.zeros((len(robot.selected_joint_dofids)))
        n_contacts = collision_result.numContacts()
        for contact_i in range(n_contacts):
            contact = collision_result.getContact(contact_i)
            dist += (-contact.penetration_depth)
            # * compute jacobian
            # self collision:
            # d g(q) / d q = 1/g(q)*(J_p1(q) - J_p2(q))
            # NOTE: the point p1 and p2 are in the world frame
            # it seems that the contact point is the intermediate point between the two objects
            normal = np.array(contact.normal)
            pt = np.array(contact.pos)
            print('penetration: ', contact.penetration_depth)
            p1 = pt - normal * 0.5 * (-contact.penetration_depth)
            p2 = pt + normal * 0.5 * (-contact.penetration_depth)
            # normal: p2 - p1
            # get the point on the link
            p1_local = pose1_inv[:3,:3]@p1 + pose1_inv[:3,3]
            p2_local = pose2_inv[:3,:3]@p2 + pose2_inv[:3,3]
            contact_pts_local.append((p1_local, p2_local))
            if link2 != "scene":
                jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                jac2 = robot.get_point_on_link_spatial_jacobian(link2, p2_local)
                jac_i = np.tensordot(normal, jac2 - jac1, axes=1)
                # jac_i = normal@(jac1 - jac2)
            else:
                # collision with environment:
                # d g(q) / d q = 1/g(q)*J_p1(q)
                jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                jac_i = np.tensordot(normal, -jac1, axes=1)
                # jac_i = normal@jac1
            jac += jac_i

        dist = dist / n_contacts
        jac = jac / n_contacts

        constrs.append(dist)
        jacs.append(jac)
        # contact_local_pts_i.append((link1, link2, obj_idx1, obj_idx2, contact_pts_local))
        contact_local_pts.append((link1, link2, obj_idx1, obj_idx2, contact_pts_local))
    constrs = np.array(constrs)
    jacs = np.array(jacs)
    return constrs, jacs, contact_local_pts

In [14]:
def compute_pt_distance_single_step(solver, contact_local_pts, q):
    # given the local pts in the world frame, and compute the distance between each pair
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.1

    res = []
    # contact_pts_local: [(pt1_on_link1, pt2_on_link2)]
    robot.set_selected_joint_values(q)
    for contact_local_pts_i_contact_k in contact_local_pts:
        link1, link2, obj_idx1, obj_idx2, contact_pts_local = contact_local_pts_i_contact_k
        pose1 = robot.get_link_pose(link1)
        pose2 = np.eye(4)
        if link2 != "scene":
            pose2 = robot.get_link_pose(link2)
        dist_avg = 0.0
        # compute the distance between each pair points
        for j in range(len(contact_pts_local)):
            p1_local, p2_local = contact_pts_local[j]
            p1 = pose1[:3,:3]@p1_local + pose1[:3,3]
            p2 = pose2[:3,:3]@p2_local + pose2[:3,3]
            dist = np.linalg.norm(p1-p2)
            dist_avg += dist
        dist_avg = dist_avg / len(contact_pts_local)
        res.append(dist_avg)

    return np.array(res)

In [15]:
# test if the compute_pt_distance_single_step is correct
q = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1])
constrs, jacs, contact_local_pts = compute_collision_constraints_single_step(solver, q)
dist = compute_pt_distance_single_step(solver, contact_local_pts, q)
print('diff: ', np.linalg.norm(constrs-dist))
# verified: the distance is correct
print('q: ')
print(q)
robot.set_selected_joint_values(q)
scene.visualize()

penetration:  -0.09820285817577967
penetration:  -0.09820285817577967
penetration:  -0.09462986957443655
penetration:  -0.09462986957443655
penetration:  -0.09750600452874501
penetration:  -0.09750600452874501
penetration:  -0.09988751833968137
penetration:  -0.09988751833968137
penetration:  -0.09680733405962999
penetration:  -0.09680733405962999
penetration:  -0.09558317121439783
penetration:  -0.09558317121439783
penetration:  -0.09979128333481348
penetration:  -0.09979128333481348
penetration:  -0.09858811860823119
penetration:  -0.09858811860823119
penetration:  -0.09979227755506119
penetration:  -0.09979227755506119
penetration:  -0.09675203634685269
penetration:  -0.09675203634685269
penetration:  -0.0981276535648439
penetration:  -0.0981276535648439
penetration:  -0.074007253737288
penetration:  -0.074007253737288
penetration:  -0.07483689246402905
penetration:  -0.07483689246402905
penetration:  -0.09978879964855403
penetration:  -0.09978879964855403
penetration:  -0.099921755

In [16]:
constrs, jacs, contact_local_pts = compute_collision_constraints_single_step(solver, qs[-1])
# finite differencing
numerical_jacs = naive_finite_diff_jacobian(lambda q: compute_pt_distance_single_step(solver, contact_local_pts, q), qs[-1], eps=1e-5)
print('constrs.shape: ', constrs.shape)
print('jacs.shape: ', jacs.shape)
print('numerical_jacs.shape: ', numerical_jacs.shape)
print('jacs diff: ', np.linalg.norm(jacs-numerical_jacs))


penetration:  -0.09820285817577967
penetration:  -0.09820285817577967
penetration:  -0.09462986957443655
penetration:  -0.09462986957443655
penetration:  -0.09750600452874501
penetration:  -0.09750600452874501
penetration:  -0.09988751833968137
penetration:  -0.09988751833968137
penetration:  -0.09680733405962999
penetration:  -0.09680733405962999
penetration:  -0.09558317121439783
penetration:  -0.09558317121439783
penetration:  -0.09979128333481348
penetration:  -0.09979128333481348
penetration:  -0.09858811860823119
penetration:  -0.09858811860823119
penetration:  -0.09979227755506119
penetration:  -0.09979227755506119
penetration:  -0.09675203634685269
penetration:  -0.09675203634685269
penetration:  -0.0981276535648439
penetration:  -0.0981276535648439
penetration:  -0.074007253737288
penetration:  -0.074007253737288
penetration:  -0.07483689246402905
penetration:  -0.07483689246402905
penetration:  -0.09978879964855403
penetration:  -0.09978879964855403
penetration:  -0.099921755

In [17]:
print(jacs[0])
print(numerical_jacs[0])

[0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0.]


In [18]:
# check the collision constraints
# first obtain the points in the link, record it, and then change the robot configuration, and get the distance of the links again.
def compute_collision_constraints(solver, qs):
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.1
    constrs = []
    jacs = []

    contact_local_pts = []  # ([link1, link2, obj_idx1, obj_idx2, contact_pts_local])
    # contact_pts_local: [(pt1_on_link1, pt2_on_link2)]
    for i in range(solver.n_waypoints):
        contact_local_pts_i = []
        robot.set_selected_joint_values(qs[i])
        # handle self collision
        collision_results = scene.compute_collision_total(security_margin=cc_margin, full=True)
        # if not in collision, then return the min distance
        for col_i in range(len(collision_results)):
            link1, link2, obj_idx1, obj_idx2, collision_result = collision_results[col_i]
            if not collision_result.isCollision():
                constrs.append(cc_margin)
                # continue
            # obtain the pose of the two links
            pose1 = robot.get_link_pose(link1)
            pose1_inv = np.linalg.inv(pose1)
            pose2 = np.eye(4)
            pose2_inv = np.eye(4)
            if link2 != "scene":
                pose2 = robot.get_link_pose(link2)
                pose2_inv = np.linalg.inv(pose2)
            contact_pts_local = []
            # otherwise, compute jacobian. use the mean of each contact point
            dist = 0.0
            # jac = np.zeros((robot.nv, 1))
            jac  = np.zeros((solver.n_waypoints, len(robot.selected_joint_dofids)))
            n_contacts = collision_result.numContacts()
            for contact_i in range(n_contacts):
                contact = collision_result.getContact(contact_i)
                dist += (-contact.penetration_depth)
                # * compute jacobian
                # self collision:
                # d g(q) / d q = 1/g(q)*(J_p1(q) - J_p2(q))
                # NOTE: the point p1 and p2 are in the world frame
                # it seems that the contact point is the intermediate point between the two objects
                normal = np.array(contact.normal)
                pt = np.array(contact.pos)
                p1 = pt - normal * 0.5 * (-contact.penetration_depth)
                p2 = pt + normal * 0.5 * (-contact.penetration_depth)
                # get the point on the link
                p1_local = pose1_inv[:3,:3]@p1 + pose1_inv[:3,3]
                p2_local = pose2_inv[:3,:3]@p2 + pose2_inv[:3,3]
                contact_pts_local.append((p1_local, p2_local))
                if link2 != "scene":
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac2 = robot.get_point_on_link_spatial_jacobian(link2, p2_local)
                    jac_i = np.tensordot(normal, jac2 - jac1, axes=1)
                    # jac_i = normal@(jac1 - jac2)
                else:
                    # collision with environment:
                    # d g(q) / d q = 1/g(q)*J_p1(q)
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac_i = np.tensordot(normal, -jac1, axes=1)
                    # jac_i = normal@jac1
                jac[i] += jac_i

            if n_contacts > 0:
                dist = dist / n_contacts
                jac[i] = jac[i] / n_contacts
            constrs.append(dist)
            jacs.append(jac)
            contact_local_pts_i.append((link1, link2, obj_idx1, obj_idx2, contact_pts_local))
        contact_local_pts.append(contact_local_pts_i)
    constrs = np.array(constrs)
    jacs = np.array(jacs)
    return constrs, jacs, contact_local_pts

In [19]:
constrs, jacs, contact_local_pts = compute_collision_constraints(solver, qs)


In [20]:
jacs.shape

(3850, 10, 7)

In [21]:
solver.compute_collision_constraints(qs)
solver_jacs = solver.prev_collision_jacs
solver_jacs.shape

len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385


(3850, 70)

In [22]:
solver_jacs.shape
print(np.linalg.norm(jacs-solver_jacs.toarray().reshape(jacs.shape)))

0.0


In [23]:
print(solver.prev_collision_constrs.shape)

(3850,)


In [24]:
solver.position_distance_constraints(qs)

array([0.44990753, 0.44990753, 0.44990753, 0.44990753, 0.44990753,
       0.44990753, 0.44990753, 0.44990753, 0.44990753, 0.44990753])

In [25]:
print(qs[1]-qs[0])
print(np.linalg.norm(qs[1]-qs[0]))
diff = np.diff(qs, axis=0)

[ 0.03537411  0.0068355   0.30589154  0.18118891  0.15315554  0.20710127
 -0.09149931]
0.4499075340776491


In [26]:
num_jac = naive_finite_diff_jacobian(lambda qs: solver.position_distance_constraints(qs), qs, eps=1e-8)
jac = solver.position_distance_jacobian(qs).toarray()
print('diff: ', np.linalg.norm(jac-num_jac.reshape(jac.shape)))

diff:  2.1696655203727744e-08


In [27]:
def position_distance_jacobian(solver, qs):
    """
    return the jacobian of the position distance constraints.
    d_pd_constrs[0] / d q[0] = (q[0]-q0)/||q[0]-q0||
    d_pd_constrs[i] / d q[i] = (q[i]-q[i-1])/||q[i]-q[i-1]||
    d_pd_constrs[i] / d q[i-1] = (q[i]-q[i-1])/||q[i]-q[i-1]||*(-1)

    J[0, 0:ndof] = (q[0]-q0)/||q[0]-q0||
    J[i, i*ndof:(i+1)*ndof] = (q[i]-q[i-1])/||q[i]-q[i-1]||
    J[i, (i-1)*ndof:i*ndof] = (q[i]-q[i-1])/||q[i]-q[i-1]||*(-1)
    TODO: verify this is correct
    """
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    diff = np.diff(qs, n=1, axis=0)
    # shape: (self.n_waypoints-1) x self.ndof
    # diff[i] = (q_i-q_{i-1})/||q_{i}-q_{i-1}||
    diff0 = qs[0] - solver.start_q
    diff0 = diff0 / np.linalg.norm(diff0)
    diff = diff / np.linalg.norm(diff, ord=2, axis=1)[:, np.newaxis] 
    indices_row = np.arange(solver.n_waypoints)[:, np.newaxis]
    indices_col = np.arange(solver.ndof).reshape((1,-1))
    row = np.zeros((solver.n_waypoints, 2*solver.ndof)) + indices_row
    col = np.zeros((solver.n_waypoints, 2*solver.ndof))
    col[1:, :solver.ndof] = (row[1:, :solver.ndof]-1) * solver.ndof + indices_col
    col[:, solver.ndof:] = (row[:, solver.ndof:]) * solver.ndof + indices_col
    data = np.zeros((solver.n_waypoints, 2*solver.ndof))
    data[0, solver.ndof:] = diff0
    data[1:, solver.ndof:] = diff
    data[1:, :solver.ndof] = -diff
    print('row: ')
    print(row)
    print('col: ')
    print(col)
    # for the first waypoint, ignore the first ndof cols
    row = row.flatten()[solver.ndof:]
    col = col.flatten()[solver.ndof:]
    data = data.flatten()[solver.ndof:]
    return sp.sparse.coo_array((data, (row, col)), shape=(solver.n_waypoints, solver.n_waypoints*solver.ndof))


In [28]:
jac = position_distance_jacobian(solver,qs)


row: 
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
 [3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.]
 [4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.]
 [5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5.]
 [6. 6. 6. 6. 6. 6. 6. 6. 6. 6. 6. 6. 6. 6.]
 [7. 7. 7. 7. 7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]
 [8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.]
 [9. 9. 9. 9. 9. 9. 9. 9. 9. 9. 9. 9. 9. 9.]]
col: 
[[ 0.  0.  0.  0.  0.  0.  0.  0.  1.  2.  3.  4.  5.  6.]
 [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13.]
 [ 7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18. 19. 20.]
 [14. 15. 16. 17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27.]
 [21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34.]
 [28. 29. 30. 31. 32. 33. 34. 35. 36. 37. 38. 39. 40. 41.]
 [35. 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47. 48.]
 [42. 43. 44. 45. 46. 47. 48. 49. 50. 51. 52. 53. 54. 55.]
 [49. 50. 51. 52. 53. 54. 55. 56. 57. 58. 59. 60. 61. 62.]
 [56. 

In [29]:
print('diff: ', np.linalg.norm(jac-num_jac.reshape(jac.shape)))

diff:  2.1696655203727744e-08


In [30]:
res = solver.joint_limit_constraints(qs)
cl = solver.cl[-len(qs.flatten()):].reshape(qs.shape)
cu = solver.cu[-len(qs.flatten()):].reshape(qs.shape)


In [31]:
cl

array([[-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159],
       [-3.14159, -1.91986, -2.96706, -2.35619, -3.14159, -1.91986,
        -3.14159]])

In [32]:
cu

array([[3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159],
       [3.14159, 1.91986, 2.96706, 2.35619, 3.14159, 1.91986, 3.14159]])

In [33]:
res

array([[ 0.94293206,  1.63142524, -2.00310902, -0.81702703, -1.60400153,
        -1.1750579 , -0.09178892],
       [ 0.97830616,  1.63826074, -1.69721748, -0.63583812, -1.45084598,
        -0.96795663, -0.18328823],
       [ 1.01368027,  1.64509624, -1.39132594, -0.45464921, -1.29769044,
        -0.76085536, -0.27478754],
       [ 1.04905438,  1.65193173, -1.0854344 , -0.27346029, -1.1445349 ,
        -0.55375409, -0.36628685],
       [ 1.08442848,  1.65876723, -0.77954286, -0.09227138, -0.99137935,
        -0.34665282, -0.45778617],
       [ 1.11980259,  1.66560273, -0.47365131,  0.08891753, -0.83822381,
        -0.13955155, -0.54928548],
       [ 1.15517669,  1.67243823, -0.16775977,  0.27010645, -0.68506826,
         0.06754972, -0.64078479],
       [ 1.1905508 ,  1.67927372,  0.13813177,  0.45129536, -0.53191272,
         0.27465099, -0.7322841 ],
       [ 1.22592491,  1.68610922,  0.44402331,  0.63248427, -0.37875718,
         0.48175226, -0.82378341],
       [ 1.26129901,  1.6929

In [34]:
solver.joint_limit_jacobian(qs.flatten()).toarray().shape

(70, 70)

In [35]:
solver.compute_collision_constraints(qs.flatten())

len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385
len(collision_results) 385


In [36]:
solver.jacobian(qs.flatten()).shape

(3930, 70)

In [37]:
solver.cl.shape

(3480,)

In [38]:
print(solver.robot.col_pair_num)
print(solver.scene.col_pair_num)

310
340


In [40]:
solver.collision_jacobian(qs.flatten()).shape
# position_jac = self.position_distance_jacobian(x)
# joint_limit_jac = self.joint_limit_jacobian(x)

(3850, 70)